In [13]:
import pandas as pd
import glob
import os

folder_path = r"E:\CAGI_data\annotateAll"

# chr*.gz 파일 전부 찾기
file_list = glob.glob(os.path.join(folder_path, "dbNSFP5.1_nsSNV.chr*.gz"))

# DataFrame 리스트 생성
dfs = []
for file in file_list:
    print(f"Reading {os.path.basename(file)} ...")
    df_temp = pd.read_csv(file, sep='\t', compression='gzip', low_memory=False)
    dfs.append(df_temp)

Reading dbNSFP5.1_nsSNV.chr1.gz ...
Reading dbNSFP5.1_nsSNV.chr10.gz ...
Reading dbNSFP5.1_nsSNV.chr11.gz ...
Reading dbNSFP5.1_nsSNV.chr12.gz ...
Reading dbNSFP5.1_nsSNV.chr13.gz ...
Reading dbNSFP5.1_nsSNV.chr14.gz ...
Reading dbNSFP5.1_nsSNV.chr15.gz ...
Reading dbNSFP5.1_nsSNV.chr16.gz ...
Reading dbNSFP5.1_nsSNV.chr17.gz ...
Reading dbNSFP5.1_nsSNV.chr18.gz ...
Reading dbNSFP5.1_nsSNV.chr19.gz ...
Reading dbNSFP5.1_nsSNV.chr2.gz ...
Reading dbNSFP5.1_nsSNV.chr20.gz ...
Reading dbNSFP5.1_nsSNV.chr21.gz ...
Reading dbNSFP5.1_nsSNV.chr22.gz ...
Reading dbNSFP5.1_nsSNV.chr3.gz ...
Reading dbNSFP5.1_nsSNV.chr4.gz ...
Reading dbNSFP5.1_nsSNV.chr5.gz ...
Reading dbNSFP5.1_nsSNV.chr6.gz ...
Reading dbNSFP5.1_nsSNV.chr7.gz ...
Reading dbNSFP5.1_nsSNV.chr8.gz ...
Reading dbNSFP5.1_nsSNV.chr9.gz ...
Reading dbNSFP5.1_nsSNV.chrM.gz ...
Reading dbNSFP5.1_nsSNV.chrX.gz ...
Reading dbNSFP5.1_nsSNV.chrY.gz ...


In [ ]:
import pandas as pd

am_cols = [
    "CHROM", "POS", "REF", "ALT", "genome",
    "uniprot_id", "transcript_id", "protein_variant",
    "am_pathogenicity", "am_class"
]

am = pd.read_csv(
    r"E:\CAGI_data\AlphaMissense_hg38.tsv.gz",
    sep="\t",
    compression="infer",
    comment="#",
    names=am_cols,
    header=0,  # 첫 줄을 header로 쓸지 여부, 보통 0
    usecols=["uniprot_id", "protein_variant"]
)

print(f"✅ 파일 로드 완료: {am.shape[0]} rows")

# --- protein_variant 파싱 (벡터화) ---
df_clean = pd.DataFrame({
    "UniProtID": am["uniprot_id"],
    "WT": am["protein_variant"].str[0],
    "MutPos": am["protein_variant"].str[1:-1].astype(int),
    "Mut": am["protein_variant"].str[-1]
})

# --- 중복 제거 ---
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
  
print(f"✅ 변환 완료: {df_clean.shape[0]} unique rows")

# --- 저장 ---
out_path = r"E:\CAGI_data\AlphaMissense_variants.tsv"
df_clean.to_csv(out_path, sep="\t", index=False)
print(f"💾 Saved to {out_path}")


✅ 파일 로드 완료: 71697555 rows
✅ 변환 완료: 63692781 unique rows
💾 Saved to E:\CAGI_data\AlphaMissense_variants.tsv


In [4]:
import json, gzip
from Bio import SeqIO

# ====== 1. JSON 로드 ======
json_path = r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\UniProtID_to_seq.json"
with open(json_path, "r") as f:
    id_to_seq = json.load(f)

# ====== 2. JSON 매핑 확인 ======
unique_ids = set(df_clean["UniProtID"].unique())
unmatched_ids = sorted([uid for uid in unique_ids if uid not in id_to_seq])
print(f"❗ JSON에서 매핑 안 된 ID 개수: {len(unmatched_ids)}")

# ====== 3. trembl fasta에서 unmatched 검색 ======
trembl_path = r"E:\CAGI_data\uniprot_trembl.fasta.gz"
target_ids = set(unmatched_ids)
found = {}

with gzip.open(trembl_path, "rt") as handle:
    for record in SeqIO.parse(handle, "fasta"):
        uid = record.id.split("|")[1] if "|" in record.id else record.id.split()[0]
        if uid in target_ids:
            found[uid] = str(record.seq)
            print(f"✅ Found in trembl: {uid}")
        if len(found) == len(target_ids):
            break

# ====== 4. JSON 업데이트 ======
if found:
    id_to_seq.update(found)
    with open(json_path, "w") as f:
        json.dump(id_to_seq, f)
    print(f"\n📦 JSON 업데이트 완료: {len(found)}개 추가됨")
else:
    print("\n⚠️ trembl에서 추가로 찾은 ID 없음")

# ====== 5. 최종 invalid_ids 확정 ======
invalid_ids = target_ids - found.keys()
print(f"❗ 최종 invalid_ids 개수: {len(invalid_ids)}")

# ====== 6. df_clean에서 invalid 제거 ======
before = df_clean.shape[0]
df_clean = df_clean[~df_clean["UniProtID"].isin(invalid_ids)].reset_index(drop=True)
after = df_clean.shape[0]

print(f"✅ df_clean 필터링 완료: {before} → {after} rows (removed {before-after})")

# ====== 7. 저장 ======
out_path = r"E:\CAGI_data\AlphaMissense_variants_filtered.tsv"
df_clean.to_csv(out_path, sep="\t", index=False)
print(f"💾 Saved filtered variants to {out_path}")


❗ JSON에서 매핑 안 된 ID 개수: 111
✅ Found in trembl: A0N4Z8
✅ Found in trembl: V9GZ13
✅ Found in trembl: A0A0G2JLJ8
✅ Found in trembl: A0A0J9YXN1
✅ Found in trembl: A0A0A0MTA2
✅ Found in trembl: A0N4Z3
✅ Found in trembl: A0A2R8Y4M2
✅ Found in trembl: A0A075B7D0
✅ Found in trembl: A0A1Y8EKQ5
✅ Found in trembl: A0A590UJ96
✅ Found in trembl: A0A0U1RQB9
✅ Found in trembl: A0A3B3IT34
✅ Found in trembl: A0A0B4J1T7
✅ Found in trembl: A0N4X5
✅ Found in trembl: A0A0J9YWD0
✅ Found in trembl: B7ZLF3
✅ Found in trembl: Q9HB66
✅ Found in trembl: A0A1Y8EI39
✅ Found in trembl: A0A087WW49
✅ Found in trembl: A0A0J9YW22
✅ Found in trembl: A0N4Z7
✅ Found in trembl: A0A0B4J2B8
✅ Found in trembl: A0A087WZ39
✅ Found in trembl: A0A1W2PNU3
✅ Found in trembl: A0N4X2
✅ Found in trembl: A0A2R8Y556
✅ Found in trembl: A0A2R8Y747
✅ Found in trembl: A0A0C4DH90
✅ Found in trembl: A0A075B7B6
✅ Found in trembl: A0A075B6H5
✅ Found in trembl: A0A0A0MT99
✅ Found in trembl: A0A075B7F1
✅ Found in trembl: A0A1B0GUZ9
✅ Found in trem

In [8]:
len(df_clean["UniProtID"].unique())

19071

In [9]:
df_clean

,UniProtID,WT,MutPos,Mut
0,Q8NH21,V,2,L
1,Q8NH21,V,2,M
2,Q8NH21,V,2,A
3,Q8NH21,V,2,E
4,Q8NH21,V,2,G
...,...,...,...,...
63611832,Q01113,F,521,L
63611833,Q01113,F,521,V
63611834,Q01113,F,521,C
63611835,Q01113,F,521,S


In [15]:
df_clean

,UniProtID,WT,MutPos,Mut
0,Q8NH21,V,2,L
1,Q8NH21,V,2,M
2,Q8NH21,V,2,A
3,Q8NH21,V,2,E
4,Q8NH21,V,2,G
...,...,...,...,...
63611832,Q01113,F,521,L
63611833,Q01113,F,521,V
63611834,Q01113,F,521,C
63611835,Q01113,F,521,S


In [16]:
import pandas as pd

# 원본과 실패 데이터 로드
df_clean = pd.read_csv(r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\AlphaMissense_variants_filtered.tsv", sep="\t")
fail_df = pd.read_csv(r"E:\CAGI_data\failed_fasta_match.tsv", sep="\t")

# 실패 ID 집합 만들기 (UniProtID, MutPos, WT 기준)
fail_keys = set(zip(fail_df["UniProtID"], fail_df["MutPos"], fail_df["WT"]))

# df_clean에서 해당 row 제거
mask = [(uid, pos, wt) not in fail_keys 
        for uid, pos, wt in zip(df_clean["UniProtID"], df_clean["MutPos"], df_clean["WT"])]

df_filtered = df_clean[mask].reset_index(drop=True)

print(f"원본 {len(df_clean)} rows → 필터링 후 {len(df_filtered)} rows")
print(f"제거된 수: {len(df_clean) - len(df_filtered)}")

# 저장
out_path = r"E:\CAGI_data\AlphaMissense_variants_passed.tsv"
df_filtered.to_csv(out_path, sep="\t", index=False)
print(f"💾 Saved filtered variants to {out_path}")

원본 63611837 rows → 필터링 후 63009766 rows
제거된 수: 602071
💾 Saved filtered variants to E:\CAGI_data\AlphaMissense_variants_passed.tsv


In [17]:
import pandas as pd

# 원본과 실패 데이터 로드
df_clean = pd.read_csv(
    r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\AlphaMissense_variants_filtered.tsv",
    sep="\t"
)
fail_df = pd.read_csv(r"E:\CAGI_data\failed_fasta_match.tsv", sep="\t")

# 실패한 UniProtID 집합
bad_uids = set(fail_df["UniProtID"])

# UID 단위로 제거
df_filtered = df_clean[~df_clean["UniProtID"].isin(bad_uids)].reset_index(drop=True)

print(f"원본 {len(df_clean)} rows → 필터링 후 {len(df_filtered)} rows")
print(f"제거된 수: {len(df_clean) - len(df_filtered)}")
print(f"제거된 UniProtID 수: {len(bad_uids)}")

# 저장
out_path = r"E:\CAGI_data\AlphaMissense_variants_passed_uid_removed.tsv"
df_filtered.to_csv(out_path, sep="\t", index=False)
print(f"💾 Saved filtered variants to {out_path}")


원본 63611837 rows → 필터링 후 62182562 rows
제거된 수: 1429275
제거된 UniProtID 수: 336
💾 Saved filtered variants to E:\CAGI_data\AlphaMissense_variants_passed_uid_removed.tsv


In [21]:
df_final = pd.read_csv(r"E:\CAGI_data\AlphaMissense_variants_passed.tsv", sep="\t")

In [22]:
df_final

,UniProtID,WT,MutPos,Mut
0,Q8NH21,V,2,L
1,Q8NH21,V,2,M
2,Q8NH21,V,2,A
3,Q8NH21,V,2,E
4,Q8NH21,V,2,G
...,...,...,...,...
63009761,Q01113,F,521,L
63009762,Q01113,F,521,V
63009763,Q01113,F,521,C
63009764,Q01113,F,521,S


In [23]:
import json
import os
import pandas as pd

# 경로 설정
json_path = r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\UniProtID_to_seq.json"
tsv_path = r"E:\CAGI_data\AlphaMissense_variants_passed.tsv"
output_dir = r"E:\CAGI_data\fasta_files_all_missense"
os.makedirs(output_dir, exist_ok=True)

# JSON 불러오기
with open(json_path, "r") as f:
    id_to_seq = json.load(f)

# TSV 불러오기
df = pd.read_csv(tsv_path, sep="\t")
# 필요한 ID 추출
unique_ids = df["UniProtID"].unique()

# 저장 및 매핑되지 않은 ID 추적
unmatched_ids = []

for uid in unique_ids:
    if uid in id_to_seq:
        fasta_path = os.path.join(output_dir, f"{uid}.fasta")
        with open(fasta_path, "w") as f:
            f.write(f">{uid}\n{id_to_seq[uid]}\n")
    else:
        unmatched_ids.append(uid)

# 매핑 안 된 ID 출력
if unmatched_ids:
    print(f"\n[❗] 다음 UniProt ID는 JSON에 매핑되지 않았습니다 ({len(unmatched_ids)}개):")
    for uid in unmatched_ids:
        print(uid)
else:
    print("\n✅ 모든 UniProt ID가 JSON에 매핑되었습니다.")


✅ 모든 UniProt ID가 JSON에 매핑되었습니다.


cat fasta_files_all_missense/*.fasta > merged_all.fasta

grep -c "^>" merged_all.fasta

mmseqs createdb merged_all.fasta merged_all

mmseqs search merged_all uniref90_mmseqs merged_result_all merged_tmp_all --threads 36 -e 0.001 --max-seqs 500

mmseqs result2msa merged_all uniref90_mmseqs merged_result_all merged_msa_a3m_all \
  --msa-format-mode 5 \
  --threads 36 \
  --max-seq-id 0.95 \
  --qid 0.3 \
  --cov 0.3 \
  --filter-msa 1 \
  --filter-min-enable 100 \
  --diff 500

In [1]:
import os, gzip
import pandas as pd
from tqdm import tqdm

# 경로
template_dir = r"C:\Users\Kunny\Research\Project\BiConVarNet\AnnotateAll\templates"
am_path = r"E:\CAGI_data\AlphaMissense_hg38.tsv.gz"

# --- AlphaMissense key set 만들기 ---
am_cols = [
    "CHROM", "POS", "REF", "ALT", "genome",
    "uniprot_id", "transcript_id", "protein_variant",
    "am_pathogenicity", "am_class"
]

am_keys = set()
for chunk in tqdm(pd.read_csv(am_path, sep="\t", compression="infer", comment="#",
                              names=am_cols, header=0, chunksize=1_000_000),
                  desc="Loading AlphaMissense"):
    for row in zip(chunk["CHROM"], chunk["POS"], chunk["REF"], chunk["ALT"]):
        chrom = str(row[0]).replace("chr", "")
        am_keys.add((chrom, int(row[1]), row[2], row[3]))

print(f"✅ AlphaMissense key 개수: {len(am_keys):,}")

# --- 템플릿과 비교 ---
total_template = 0
matched = 0

for fname in sorted(os.listdir(template_dir)):
    if not fname.endswith(".template.gz"):
        continue
    path = os.path.join(template_dir, fname)

    with gzip.open(path, "rt") as f:
        for line in f:
            if not line.strip() or line.startswith("#"):
                continue
            cols = line.strip().split("\t")
            if len(cols) < 4:
                continue
            chrom, pos, ref, alt = cols[:4]
            pos = int(pos)
            total_template += 1
            if (chrom, pos, ref, alt) in am_keys:
                matched += 1

    print(f"✅ {fname}: 누적 매칭 {matched:,} / {total_template:,} ({matched/total_template:.4%})")

print("==="*10)
print(f"총 템플릿 {total_template:,}개 중 매칭 {matched:,}개 ({matched/total_template:.2%})")


Loading AlphaMissense: 72it [03:52,  3.24s/it]


✅ AlphaMissense key 개수: 71,034,268
✅ dbNSFP4_nsSNV.chr1.template.gz: 누적 매칭 7,196,208 / 8,309,694 (86.6002%)
✅ dbNSFP4_nsSNV.chr10.template.gz: 누적 매칭 10,013,898 / 11,496,631 (87.1029%)
✅ dbNSFP4_nsSNV.chr11.template.gz: 누적 매칭 14,292,665 / 16,309,032 (87.6365%)
✅ dbNSFP4_nsSNV.chr12.template.gz: 누적 매칭 17,926,559 / 20,576,718 (87.1206%)
✅ dbNSFP4_nsSNV.chr13.template.gz: 누적 매칭 19,192,445 / 22,067,264 (86.9725%)
✅ dbNSFP4_nsSNV.chr14.template.gz: 누적 매칭 21,403,981 / 24,636,307 (86.8798%)
✅ dbNSFP4_nsSNV.chr15.template.gz: 누적 매칭 23,925,693 / 27,466,289 (87.1093%)
✅ dbNSFP4_nsSNV.chr16.template.gz: 누적 매칭 26,919,553 / 30,908,012 (87.0957%)
✅ dbNSFP4_nsSNV.chr17.template.gz: 누적 매칭 31,046,642 / 35,581,589 (87.2548%)
✅ dbNSFP4_nsSNV.chr18.template.gz: 누적 매칭 32,186,250 / 36,879,899 (87.2732%)
✅ dbNSFP4_nsSNV.chr19.template.gz: 누적 매칭 36,867,745 / 42,271,014 (87.2176%)
✅ dbNSFP4_nsSNV.chr2.template.gz: 누적 매칭 42,126,406 / 48,287,990 (87.2399%)
✅ dbNSFP4_nsSNV.chr20.template.gz: 누적 매칭 43,836,519 / 50,